In [1]:
from sklearn.model_selection import LeaveOneOut, GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from matplotlib import style
from sklearn.svm import SVC
import seaborn as sns
import pandas as pd
import scienceplots
import numpy as np

In [2]:
plt.style.use(['science', 'nature'])

In [3]:
data = pd.read_csv('../stats/cleaned_data.csv')

In [4]:
X = data.drop(['subject', 'group'], axis=1)
y = data['group'].apply(lambda x: 0 if x == 'hc' else 1)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

In [6]:
class ModelEvaluator:

    def __init__(self, model, X_train, X_test, y_train, y_test, param_grid):

        self.__X_train, self.__X_test, self.__y_train, self.__y_test = X_train, X_test, y_train, y_test
        self.__cv = LeaveOneOut()
        self.__param_grid = param_grid

        self.__pipe = Pipeline(steps=[('scaler', StandardScaler()),
                                    ('pca', PCA(n_components=0.8)),
                                    ('model', model)])

    def evaluate(self):

        grid = GridSearchCV(self.__pipe, self.__param_grid, cv=self.__cv)
        grid.fit(self.__X_train, self.__y_train)

        print(f'Best parameters: {grid.best_params_}')

        y_pred = grid.predict(self.__X_test)
        cm = confusion_matrix(self.__y_test, y_pred)

        tn, fp, fn, tp = cm.ravel()
        sensitivity = tp / (tp + fn)
        specificity = tn / (tn + fp)

        print(f'Accuracy: {grid.score(self.__X_test, self.__y_test)}')
        print(f'Sensitivity: {sensitivity}')
        print(f'Specificity: {specificity}')

        plt.figure(figsize=(15, 6))
        plt.subplot(1, 2, 1)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['HC', 'FCD'], yticklabels=['HC', 'FCD'])
        plt.xlabel('Predicted')
        plt.ylabel('Actual')

        best_model = grid.best_estimator_
        train_sizes, train_scores, test_scores = learning_curve(
            best_model, self.__X_train, self.__y_train, cv=self.__cv, train_sizes=np.linspace(.1, 1.0, 5)
        )

        train_scores_mean = np.mean(train_scores, axis=1)
        train_scores_std = np.std(train_scores, axis=1)
        test_scores_mean = np.mean(test_scores, axis=1)
        test_scores_std = np.std(test_scores, axis=1)

        plt.subplot(1, 2, 2)
        plt.title('Learning Curve')
        plt.xlabel('Training examples')
        plt.ylabel('Score')
        plt.grid()

        plt.fill_between(train_sizes, train_scores_mean - train_scores_std, train_scores_mean + train_scores_std, alpha=0.1, color='r')
        plt.fill_between(train_sizes, test_scores_mean - test_scores_std, test_scores_mean + test_scores_std, alpha=0.1, color='g')

        plt.plot(train_sizes, train_scores_mean, 'o-', color='r', label='Training score')
        plt.plot(train_sizes, test_scores_mean, 'o-', color='g', label='Cross-validation score')
        
        plt.legend(loc='best')
        plt.show()

In [ ]:
model =  GradientBoostingClassifier(random_state=42)
evaluator_gradient = ModelEvaluator(model, X_train, X_test, y_train, y_test, {'model__learning_rate': [0.1, 0.01, 0.001], 'model__n_estimators': [100, 200, 300]})
evaluator_gradient.evaluate()

In [ ]:
model =  SVC(random_state=42)
evaluator_svc = ModelEvaluator(model, X_train, X_test, y_train, y_test, {'model__kernel': ['linear', 'sigmoid', 'rbf'], 'model__C': [1, 10, 100]})
evaluator_svc.evaluate()

In [ ]:
model =  LogisticRegression(max_iter=300, random_state=42)
evaluator_logistic = ModelEvaluator(model, X_train, X_test, y_train, y_test, {'model__C': [0.1, 1, 10], 'model__tol': [0.01, 0.001, 0.0001]})
evaluator_logistic.evaluate()

In [ ]:
model =  RandomForestClassifier(random_state=42)
evaluator_forest = ModelEvaluator(model, X_train, X_test, y_train, y_test, {'model__n_estimators': [100, 200, 300], 'model__max_depth': [3, 5, 7]})
evaluator_forest.evaluate()